# Deepfake detection (Xception) — Colab subset training

This notebook trains **Xception** on a **subset** of `dataset/real` + `dataset/fake` using **Colab GPU (CUDA)**.

It saves:
- `progress_log.csv` and `progress_diff_log.txt` (mentor-friendly progress)
- `training_curves.png`
- `confusion_matrix_test.png`
- `final_report.json`


In [ ]:
!nvidia-smi
!pip -q install torch torchvision pytorchcv albumentations opencv-python-headless scikit-learn matplotlib tqdm pillow
import torch
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
import os, random, json
from pathlib import Path

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms

from pytorchcv.model_provider import get_model as ptcv_get_model

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

import matplotlib.pyplot as plt
from tqdm import tqdm


CLASS_NAMES = ("real", "fake")  # 0=real, 1=fake

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class FaceDeepFakeDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = np.array(Image.open(self.paths[idx]).convert("RGB"))
        img = Image.fromarray(img)
        if self.transform is not None:
            img = self.transform(img)
        return img, int(self.labels[idx])


def collect_paths(data_dir: str):
    data_dir = Path(data_dir)
    real_dir = data_dir / "real"
    fake_dir = data_dir / "fake"

    exts = {".jpg", ".jpeg", ".png"}
    real = sorted([p for p in real_dir.iterdir() if p.suffix.lower() in exts])
    fake = sorted([p for p in fake_dir.iterdir() if p.suffix.lower() in exts])

    paths = real + fake
    labels = [0] * len(real) + [1] * len(fake)
    if len(paths) == 0:
        raise FileNotFoundError("No images found under dataset/real and dataset/fake")
    return paths, labels


def subset_max_per_class(paths, labels, max_per_class, seed=42):
    rng = np.random.default_rng(seed)
    labels_np = np.array(labels)
    paths_np = np.array(paths, dtype=object)
    keep = []
    for c in [0, 1]:
        idx = np.where(labels_np == c)[0]
        take = min(max_per_class, len(idx))
        keep.extend(rng.choice(idx, size=take, replace=False).tolist())
    keep = sorted(keep)
    return paths_np[keep].tolist(), labels_np[keep].tolist()


def stratified_split(paths, labels, seed=42, train_frac=0.8, val_frac=0.1):
    X_train, X_temp, y_train, y_temp = train_test_split(
        paths, labels, test_size=(1 - train_frac), stratify=labels, random_state=seed
    )
    val_ratio = val_frac / (1 - train_frac)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=(1 - val_ratio), stratify=y_temp, random_state=seed
    )
    return X_train, X_val, X_test, y_train, y_val, y_test


def get_transforms(img_size=299, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.82, 1.0), ratio=(0.92, 1.08)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.18, contrast=0.18, saturation=0.10, hue=0.02),
        transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.2))], p=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])
    return train_tf, eval_tf


def replace_xception_head(model, num_classes=2, hidden=512, dropout=0.5):
    in_f = model.output.in_features
    model.output = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(in_f, hidden),
        nn.ReLU(inplace=True),
        nn.BatchNorm1d(hidden),
        nn.Dropout(dropout * 0.6),
        nn.Linear(hidden, num_classes),
    )
    return model


def freeze_phase1_head_only(model):
    for name, p in model.named_parameters():
        if name.startswith("output."):
            p.requires_grad = True
        else:
            p.requires_grad = False


def unfreeze_phase2_last_stages(model):
    for _, p in model.named_parameters():
        p.requires_grad = False
    for name, p in model.named_parameters():
        if name.startswith("output."):
            p.requires_grad = True
        if name.startswith("features.stage4.") or name.startswith("features.final_block."):
            p.requires_grad = True


def mixup_batch(x, y, alpha, device):
    if alpha <= 0:
        return x, y, y, 1.0
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(x.size(0), device=device)
    x_m = lam * x + (1 - lam) * x[idx]
    return x_m, y, y[idx], lam


def eval_model(model, loader, device):
    model.eval()
    all_true, all_pred = [], []
    all_prob = []

    with torch.no_grad():
        for x, y in tqdm(loader, desc="eval", leave=False):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            # TTA: horizontal flip
            logits = model(x)
            xf = torch.flip(x, dims=[3])
            logits2 = model(xf)
            logits = 0.5 * (logits + logits2)

            p_fake = torch.softmax(logits, dim=1)[:, 1]
            pred = (p_fake >= 0.5).long()

            all_true.extend(y.detach().cpu().numpy().tolist())
            all_pred.extend(pred.detach().cpu().numpy().tolist())
            all_prob.extend(p_fake.detach().cpu().numpy().tolist())

    acc = accuracy_score(all_true, all_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        all_true, all_pred, labels=[0, 1], average=None, zero_division=0
    )
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        all_true, all_pred, labels=[0, 1], average="macro", zero_division=0
    )
    cm = confusion_matrix(all_true, all_pred, labels=[0, 1])
    return {
        "loss": None,
        "acc": float(acc),
        "p_macro": float(p_macro),
        "r_macro": float(r_macro),
        "f1_macro": float(f1_macro),
        "p_per_class": prec.tolist(),
        "r_per_class": rec.tolist(),
        "f1_per_class": f1.tolist(),
        "y_true": np.array(all_true),
        "y_pred": np.array(all_pred),
        "y_prob": np.array(all_prob),
        "cm": cm,
    }


def save_confusion_matrix(cm, out_path, title="Confusion matrix"):
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(CLASS_NAMES)
    ax.set_yticklabels(CLASS_NAMES)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center", color="black")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def plot_training_curves(history, out_path):
    xs = np.arange(len(history["train_loss"]))
    fig, ax = plt.subplots(2, 1, figsize=(10, 8), constrained_layout=True)

    ax0, ax1 = ax
    ax0.plot(xs, history["train_loss"], label="train_loss", color="#2563eb")
    ax0.set_title("Training loss")
    ax0.grid(True, alpha=0.3)
    ax0.legend()

    ax1.plot(xs, history["val_acc"], label="val_acc", color="#16a34a", marker="o", markersize=3)
    ax1.plot(xs, history["val_f1_macro"], label="val_f1_macro", color="#dc2626", marker="o", markersize=3)
    ax1.set_title("Validation metrics")
    ax1.set_ylim(0, 1.05)
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc="lower right")

    fig.savefig(out_path, dpi=150)
    plt.close(fig)


In [ ]:
# ==== Run subset experiment ==== 
DATA_DIR = '/content/dataset'  # expects dataset/real and dataset/fake
OUT_DIR = '/content/deepfake_out'
MAX_PER_CLASS = 300  # subset cap for fast mentor demo
IMG_SIZE = 299
SEED = 42

EPOCHS_P1 = 2
EPOCHS_P2 = 4
BATCH_SIZE = 32
MIXUP_ALPHA = 0.2

LR_HEAD = 3e-4
LR_BACKBONE = 1e-5
LABEL_SMOOTHING = 0.08

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
set_seed(SEED)

paths, labels = collect_paths(DATA_DIR)
print('Total images:', len(paths), 'real:', labels.count(0), 'fake:', labels.count(1))

paths, labels = subset_max_per_class(paths, labels, MAX_PER_CLASS, SEED)
print('Subset images:', len(paths), 'real:', labels.count(0), 'fake:', labels.count(1))

X_train, X_val, X_test, y_train, y_val, y_test = stratified_split(paths, labels, SEED)
print('Split sizes:', len(X_train), len(X_val), len(X_test))

train_tf, eval_tf = get_transforms(IMG_SIZE)
train_ds = FaceDeepFakeDataset(X_train, y_train, train_tf)
val_ds = FaceDeepFakeDataset(X_val, y_val, eval_tf)
test_ds = FaceDeepFakeDataset(X_test, y_test, eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
amp = device.type == 'cuda'

# Model
xception = ptcv_get_model('xception', pretrained=True)
xception = replace_xception_head(xception, num_classes=2, hidden=512, dropout=0.5)
xception.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
scaler = torch.cuda.amp.GradScaler(enabled=amp)

# Mentor progress logs
csv_path = Path(OUT_DIR) / 'progress_log.csv'
diff_path = Path(OUT_DIR) / 'progress_diff_log.txt'

csv_path.write_text('epoch,phase,train_loss,val_acc,val_f1_macro,delta_val_acc,delta_val_f1\n', encoding='utf-8')
diff_path.write_text('', encoding='utf-8')

prev = None
history = {'train_loss': [], 'val_acc': [], 'val_precision_macro': [], 'val_recall_macro': [], 'val_f1_macro': []}


def train_one_epoch(model, loader, optimizer, phase_tag, epoch_idx):
    model.train()
    running = 0.0
    n = 0
    for xb, yb in tqdm(loader, desc=f'train {phase_tag} ep{epoch_idx}', leave=False):
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # MixUp
        x_mix, y_a, y_b2, lam = mixup_batch(xb, yb, MIXUP_ALPHA, device)

        with torch.cuda.amp.autocast(enabled=amp):
            logits = model(x_mix)
            if lam < 1.0:
                loss = lam * criterion(logits, y_a) + (1.0 - lam) * criterion(logits, y_b2)
            else:
                loss = criterion(logits, yb)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bs = xb.size(0)
        running += float(loss.item()) * bs
        n += bs

    return running / max(n, 1)


def log_metrics(epoch_idx, phase_tag, train_loss, metrics):
    global prev
    if prev is None:
        d_acc, d_f1 = 0.0, 0.0
    else:
        d_acc = metrics['acc'] - prev['acc']
        d_f1 = metrics['f1_macro'] - prev['f1_macro']

    history['train_loss'].append(float(train_loss))
    history['val_acc'].append(float(metrics['acc']))
    history['val_precision_macro'].append(float(metrics['p_macro']))
    history['val_recall_macro'].append(float(metrics['r_macro']))
    history['val_f1_macro'].append(float(metrics['f1_macro']))

    with open(csv_path, 'a', encoding='utf-8') as f:
        f.write(
            f"{epoch_idx},{phase_tag},{train_loss:.6f},{metrics['acc']:.6f},{metrics['f1_macro']:.6f},{d_acc:.6f},{d_f1:.6f}\n"
        )

    with open(diff_path, 'a', encoding='utf-8') as f:
        f.write(
            f"Epoch {epoch_idx} | {phase_tag} | train_loss={train_loss:.4f} | val_acc={metrics['acc']:.4f} (Δ{d_acc:+.4f}) | val_f1={metrics['f1_macro']:.4f} (Δ{d_f1:+.4f})\n"
        )

    prev = {'acc': metrics['acc'], 'f1_macro': metrics['f1_macro']}


# Phase 1: head only
freeze_phase1_head_only(xception)
phase_tag = 'p1'

for ep in range(1, EPOCHS_P1 + 1):
    # optimizer only for params with requires_grad=True
    params = [p for p in xception.parameters() if p.requires_grad]
    opt = optim.AdamW(params, lr=LR_HEAD, weight_decay=1e-4)

    tr_loss = train_one_epoch(xception, train_loader, opt, phase_tag, ep)
    m = eval_model(xception, val_loader, device)
    log_metrics(ep, phase_tag, tr_loss, m)
    print(f"P1 ep{ep}: train_loss={tr_loss:.4f} val_acc={m['acc']:.4f} val_f1={m['f1_macro']:.4f}")

    # optional: checkpoint
    torch.save({'state_dict': xception.state_dict()}, str(Path(OUT_DIR) / f'ckpt_p1_ep{ep}.pt'))


# Phase 2: unfreeze last stages
unfreeze_phase2_last_stages(xception)

# Split param groups (backbone vs head)
head_params = list(xception.output.parameters())
back_params = [p for n, p in xception.features.named_parameters() if (n.startswith('stage4') or n.startswith('final_block'))]

opt2 = optim.AdamW(
    [
        {'params': back_params, 'lr': LR_BACKBONE},
        {'params': head_params, 'lr': LR_HEAD * 0.5},
    ],
    weight_decay=1e-4,
)

sched = optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=EPOCHS_P2, eta_min=LR_BACKBONE * 0.1)
phase_tag = 'p2'

for ep2 in range(1, EPOCHS_P2 + 1):
    ep_global = EPOCHS_P1 + ep2
    tr_loss = train_one_epoch(xception, train_loader, opt2, phase_tag, ep_global)
    sched.step()

    m = eval_model(xception, val_loader, device)
    log_metrics(ep_global, phase_tag, tr_loss, m)
    print(f"P2 ep{ep2} (global {ep_global}): train_loss={tr_loss:.4f} val_acc={m['acc']:.4f} val_f1={m['f1_macro']:.4f}")

    torch.save({'state_dict': xception.state_dict()}, str(Path(OUT_DIR) / f'ckpt_p2_ep{ep2}.pt'))


# Final test + plots
final_ckpt = Path(OUT_DIR) / f'ckpt_p2_ep{EPOCHS_P2}.pt'
if final_ckpt.exists():
    xception.load_state_dict(torch.load(final_ckpt, map_location=device)['state_dict'])

mtest = eval_model(xception, test_loader, device)
print('TEST:', {k: mtest[k] for k in ['acc','f1_macro','p_macro','r_macro']})

# Curves
plot_training_curves({
    'train_loss': history['train_loss'],
    'val_acc': history['val_acc'],
    'val_precision_macro': history['val_precision_macro'],
    'val_recall_macro': history['val_recall_macro'],
    'val_f1_macro': history['val_f1_macro'],
}, str(Path(OUT_DIR) / 'training_curves.png'))

# Confusion matrix
save_confusion_matrix(mtest['cm'], str(Path(OUT_DIR) / 'confusion_matrix_test.png'), 'Confusion matrix — test')

report = {
    'subset_max_per_class': MAX_PER_CLASS,
    'test_accuracy': float(mtest['acc']),
    'test_macro_precision': float(mtest['p_macro']),
    'test_macro_recall': float(mtest['r_macro']),
    'test_macro_f1': float(mtest['f1_macro']),
    'confusion_matrix': mtest['cm'].tolist(),
}
(Path(OUT_DIR) / 'final_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')

print('DONE. Outputs in:', OUT_DIR)
